# Explore: run the GA yourself

A hands-on tour of the framework on **one kangaroo**: pick settings, run the GA, score the result with the course grader, and look at it. Select the **Python (ps1)** kernel and run the cells top to bottom; each takes seconds.

Results are saved under `runs/explore/` (git-ignored). Don't save exploration results into `submissions/`: CI checks every file there against the full submission requirements, and a one-kangaroo file would fail them.

In [ ]:
# Setup: work from the repo root so `linkopt` and `LINKS` import.
import os
from pathlib import Path

root = Path.cwd()
while not (root / 'linkopt').is_dir() and root != root.parent:
    root = root.parent
os.chdir(root)

import linkopt  # always import first: pins JAX to the CPU
import numpy as np
import matplotlib.pyplot as plt

print('working in', root)

## 1. Choose settings

Start from a preset and override anything (see the README's *Choosing settings* table). `quick` is a real but short run. Try changing `n_gen`, `pop_size` or `mutation_prob`.

In [ ]:
from linkopt.config import preset

cfg = preset('quick')          # e.g. preset('quick', n_gen=60, pop_size=100)
cfg

## 2. Run the GA on one kangaroo

- `target`: `0` = Kangaroo 1 (easiest), `1` = Kangaroo 2, `2` = Kangaroo 3 (loosest limits)
- `n_joints`: mechanism size (5 to 20)
- `seed`: different seeds give different random starting mechanisms and a different run. Results vary a lot between seeds; that's normal for a GA.

`result.designs` are the feasible non-dominated designs (inside both limits, and no other design is better at both). `result.F` holds each one's `[distance, material]`.

In [ ]:
from linkopt.ga import run_ga

TARGET = 1
result = run_ga(target=TARGET, n_joints=7, seed=0, cfg=cfg)

print(f'{len(result.designs)} designs, {result.n_evals} evaluations, {result.seconds:.1f}s')
print('best starting distance:', result.start_F[:, 0].min().round(3))
result.F  # [distance, material] per design

## 3. Score it with the course grader

`build_submission` puts the designs under `'Problem {TARGET + 1}'`; `save` checks the starter notebook's format, writes the file, and scores it with `evaluate_submission`. Only this kangaroo scores; the other two are empty (0).

In [ ]:
from linkopt.submission import build_submission, save

scores = save(build_submission({TARGET: result.designs}), f'runs/explore/kangaroo{TARGET + 1}.npy')
scores

## 4. The trade-off: distance vs. material

Each point is a design; the shaded area inside the limits box is the hypervolume (the score).

In [ ]:
from LINKS.CP import REFERENCE_POINTS
from LINKS.Visualization import GAVisualizer

if len(result.F):
    GAVisualizer().plot_HV(result.F, REFERENCE_POINTS[TARGET], objective_labels=['Distance', 'Material'])
else:
    print('no feasible designs: try another seed, more generations, or TARGET = 2')

## 5. The closest-fitting mechanism

The mechanism (fixed joints hatched, motor link yellow, traced joint highlighted), then its traced path against the kangaroo outline.

In [ ]:
from linkopt.ga import target_curve
from LINKS.Geometry import CurveEngine
from LINKS.Kinematics import MechanismSolver
from LINKS.Visualization import MechanismVisualizer

best = result.designs[np.argmin(result.F[:, 0])]
print('distance, material:', result.F[np.argmin(result.F[:, 0])])

plt.figure(figsize=(6, 6))
MechanismVisualizer()(best['x0'], best['edges'], best['fixed_joints'], best['motor'],
                      highlight=best['target_joint'], ax=plt.gca())
plt.show()

path = MechanismSolver(device='cpu')(best['x0'], best['edges'], best['fixed_joints'], best['motor'])
CurveEngine(device='cpu').visualize_comparison(path[best['target_joint']], target_curve(TARGET))

## 6. More seeds, more designs

Each seed is an independent run. Pooling their designs usually raises the hypervolume: this is what `run.py` will do at scale, across all three kangaroos.

In [ ]:
from pymoo.indicators.hv import HV

pooled, F = [], []
for seed in range(3):
    r = run_ga(target=TARGET, n_joints=7, seed=seed, cfg=cfg)
    pooled += r.designs
    F += list(r.F)
    own = HV(REFERENCE_POINTS[TARGET])(r.F) if len(r.F) else 0.0
    print(f'seed {seed}: {len(r.designs)} designs, own hypervolume {own:.3f}')

scores = save(build_submission({TARGET: pooled}), f'runs/explore/kangaroo{TARGET + 1}_pooled.npy')
print('pooled:', scores['Score Breakdown'][f'Problem {TARGET + 1}'])

## Things to try

- `TARGET = 2` (Kangaroo 3) finds feasible designs most easily; `TARGET = 0` is the strictest.
- Bigger mechanisms: `n_joints=9` or `12`. More links can trace more complex shapes.
- Longer runs: `preset('quick', n_gen=100, pop_size=100)`.
- Mutation: `preset('quick', mutation_prob=0.3)` vs `0.9`.

 From Advanced notebook hints:
- Why not do some preprocessing to random mechanisms before running GA?
- Can you use the gradients of both functions in gradient based optimization?
- Can you cycle through multiple optimization runs?
- Can you mix different kinds of GA? We showed how GA can be used for a single mechanism skeleton and how it can be used to also create the mechanisms?
- Can you modify the GA mutations and cross-over?
- Is there a more efficient representation of mechanisms we can use?
- Are there smarter gradient-based optimization methods?

Anything worth keeping moves into `linkopt/` so `run.py` and CI use it.